# Workspace Kolaborasi: CareerMatch AI
**Tim:** PJK-RM119

**Deskripsi:** Notebook utama untuk pengembangan sistem rekomendasi ATS CV menggunakan NLP.
Notebook ini dibagi berdasarkan target mingguan dan pembagian peran (Role) masing-masing anggota.

---
## INISIALISASI & SETUP LINGKUNGAN (Global)
Bagian ini berisi instalasi library dan import modul yang akan digunakan oleh seluruh tim. Pastikan untuk menjalankan cell ini pertama kali, dan update cell ini sesuai dengan kebutuhan masing masing.

In [1]:
# Install library eksternal utama
!pip install kagglehub pypdf2 spacy nltk pandas numpy scikit-learn mlflow tqdm

# Unduh model bahasa Inggris untuk SpaCy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [23]:
# Update cell ini sesuai kebutuhan
# Import seluruh library yang dibutuhkan
import os
import pandas as pd
import numpy as np
import re
import spacy
import nltk
import kagglehub
from PyPDF2 import PdfReader
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import mlflow
from google.colab import files as colab_files
import io

# Setup NLTK
nltk.download('stopwords')
nltk.download('punkt')

print("Setup Environment Selesai!")

Setup Environment Selesai!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


---
## MINGGU 1: PENGUMPULAN DATA & EKSPLORASI

### Tugas 1.1: Ekstraktor Dokumen PDF
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Membuat fungsi untuk mengekstrak teks mentah dari file PDF CV (Format ATS-Friendly).

In [3]:
def extract_pdf_text(pdf_path):
    """
    Ekstraksi teks mentah dari PDF dengan penanganan error kelas produksi.
    Fungsi ini bertanggung jawab mengubah file binary PDF menjadi raw text.
    """
    try:
        reader = PdfReader(pdf_path)
        raw_text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                raw_text += extracted + " "

        if not raw_text.strip():
            return None, "Dokumen kosong atau berbasis gambar (Tidak ATS-Friendly)."

        return raw_text, "Success"

    except Exception as e:
        return None, str(e)

### Tugas 1.2: Akuisisi Dataset Lowongan Kerja (Kaggle)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Mencari dataset *Job Postings* atau yang berkaitan yang relevan di Kaggle dan memuatnya ke dalam Pandas DataFrame.

> **Saran:** Hindari mengunduh manual. Gunakan library `kagglehub` untuk langsung menarik data ke dalam memori.

In [4]:
# Tulis kode download dataset Kaggle dan load ke dalam DataFrame (df_jobs) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Download dataset dari Kaggle
path = kagglehub.dataset_download("madhab/jobposts")

print("Path to dataset files:", path)

100%|██████████| 14.0M/14.0M [00:00<00:00, 180MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/madhab/jobposts/versions/1


In [5]:
# Melihat file yang tersedia
files = os.listdir(path)

print("Daftar file dalam dataset:")
for file in files:
    print(file)

Daftar file dalam dataset:
screenshot.jpg
data job posts.csv


In [6]:
# Ganti nama file sesuai file CSV yang muncul
csv_path = os.path.join(path, "data job posts.csv")

# Load dataset
df_jobs = pd.read_csv(csv_path)

### Tugas 1.3: Exploratory Data Analysis (EDA)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Melakukan pengecekan karakteristik data awal. Cek total baris, nama kolom, jumlah *missing values*, dan lihat sekilas isi kolom deskripsi pekerjaan.

In [7]:
# Tulis kode untuk EDA (df.head, df.info, df.isna().sum()) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Menampilkan 5 data pertama
df_jobs.head()

,jobpost,date,Title,Company,AnnouncementCode,Term,Eligibility,Audience,StartDate,Duration,...,Salary,ApplicationP,OpeningDate,Deadline,Notes,AboutC,Attach,Year,Month,IT
0,AMERIA Investment Consulting Company\r\nJOB TI...,"Jan 5, 2004",Chief Financial Officer,AMERIA Investment Consulting Company,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,"To apply for this position, please submit a\r\...",NaN,26 January 2004,NaN,NaN,NaN,2004,1,False
1,International Research & Exchanges Board (IREX...,"Jan 7, 2004",Full-time Community Connections Intern (paid i...,International Research & Exchanges Board (IREX),NaN,NaN,NaN,NaN,NaN,3 months,...,NaN,Please submit a cover letter and resume to:\r\...,NaN,12 January 2004,NaN,The International Research & Exchanges Board (...,NaN,2004,1,False
2,Caucasus Environmental NGO Network (CENN)\r\nJ...,"Jan 7, 2004",Country Coordinator,Caucasus Environmental NGO Network (CENN),NaN,NaN,NaN,NaN,NaN,Renewable annual contract\r\nPOSITION,...,NaN,Please send resume or CV toursula.kazarian@......,NaN,20 January 2004\r\nSTART DATE: February 2004,NaN,The Caucasus Environmental NGO Network is a\r\...,NaN,2004,1,False
3,Manoff Group\r\nJOB TITLE: BCC Specialist\r\n...,"Jan 7, 2004",BCC Specialist,Manoff Group,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Please send cover letter and resume to Amy\r\n...,NaN,23 January 2004\r\nSTART DATE: Immediate,NaN,NaN,NaN,2004,1,False
4,Yerevan Brandy Company\r\nJOB TITLE: Software...,"Jan 10, 2004",Software Developer,Yerevan Brandy Company,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Successful candidates should submit\r\n- CV; \...,NaN,"20 January 2004, 18:00",NaN,NaN,NaN,2004,1,True


In [8]:
# Info dataset
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19001 entries, 0 to 19000
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   jobpost           19001 non-null  object
 1   date              19001 non-null  object
 2   Title             18973 non-null  object
 3   Company           18994 non-null  object
 4   AnnouncementCode  1208 non-null   object
 5   Term              7676 non-null   object
 6   Eligibility       4930 non-null   object
 7   Audience          640 non-null    object
 8   StartDate         9675 non-null   object
 9   Duration          10798 non-null  object
 10  Location          18969 non-null  object
 11  JobDescription    15109 non-null  object
 12  JobRequirment     16479 non-null  object
 13  RequiredQual      18517 non-null  object
 14  Salary            9622 non-null   object
 15  ApplicationP      18941 non-null  object
 16  OpeningDate       18295 non-null  object
 17  Deadline    

In [9]:
# Mengecek data kosong
df_jobs.isna().sum().sort_values(ascending=False)

,0
Audience,18361
AnnouncementCode,17793
Attach,17442
Notes,16790
Eligibility,14071
Term,11325
Salary,9379
StartDate,9326
Duration,8203
AboutC,6531


---
## MINGGU 2: PRA-PEMROSESAN DATA (PREPROCESSING)

### Tugas 2.1: Data Cleaning (Pembersihan Dataset)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Menghapus data duplikat dan baris yang kolom *Job Description/Requirements*-nya kosong (*missing values*), karena data kosong akan merusak perhitungan NLP.

In [10]:
# Tulis kode pembersihan DataFrame (dropna, drop_duplicates) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Memilih kolom yang relevan
clean_df = df_jobs[[
    "Title",
    "JobDescription",
    "JobRequirment",
    "RequiredQual"
]]

In [11]:
# Mengecek missing values
clean_df.isna().sum()

,0
Title,28
JobDescription,3892
JobRequirment,2522
RequiredQual,484


In [12]:
# Menghapus baris yang kosong
clean_df = clean_df.dropna(subset=[
    "Title",
    "JobDescription",
    "JobRequirment",
    "RequiredQual"
])

In [13]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13124 entries, 0 to 19000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           13124 non-null  object
 1   JobDescription  13124 non-null  object
 2   JobRequirment   13124 non-null  object
 3   RequiredQual    13124 non-null  object
dtypes: object(4)
memory usage: 512.7+ KB


In [14]:
# Menghapus data duplikat
clean_df = clean_df.drop_duplicates()

In [15]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12379 entries, 0 to 19000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           12379 non-null  object
 1   JobDescription  12379 non-null  object
 2   JobRequirment   12379 non-null  object
 3   RequiredQual    12379 non-null  object
dtypes: object(4)
memory usage: 483.6+ KB


In [16]:
clean_df.head()

,Title,JobDescription,JobRequirment,RequiredQual
0,Chief Financial Officer,AMERIA Investment Consulting Company is seekin...,- Supervises financial management and administ...,"To perform this job successfully, an\r\nindivi..."
2,Country Coordinator,Public outreach and strengthening of a growing...,- Working with the Country Director to provide...,"- Degree in environmentally related field, or ..."
3,BCC Specialist,The LEAD (Local Enhancement and Development fo...,- Identify gaps in knowledge and overseeing in...,"- Advanced degree in public health, social sci..."
13,"Community Development, Capacity Building and C...",Food Security Regional Cooperation and Stabili...,- Assist the Tavush Marz communities and commu...,- Higher Education and/or professional experie...
17,Country Economist (NOB),The United Nations Development Programme in Ar...,The incumbent under direct supervision of UNDP...,- Minimum Masters Degree in Economics;\r\n- Mi...


### Tugas 2.2: NLP Text Preprocessing Pipeline
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Membuat satu fungsi terpusat (`clean_text`) untuk menyeragamkan teks (huruf kecil, hapus URL, hapus karakter khusus) dan melakukan Tokenisasi, Stopwords Removal, serta Lemmatization menggunakan SpaCy/NLTK. Fungsi ini akan di-apply ke dataset lowongan kerja.

In [17]:
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

universal_stopwords = {
    'summary', 'skill', 'qualification',
    'project', 'projects', 'certification', 'honor', 'award', 'contact',
    'email', 'phone', 'location', 'github', 'linkedin', 'profile',
    'status', 'expected', 'current', 'gpa', 'cumulative'
}

hr_fluff = {
    'ability', 'candidate', 'environment', 'strong', 'good',
    'hard', 'deadline', 'oriented', 'fast', 'paced', 'proven',
    'excellent', 'preferred', 'required', 'requirement'
}

stop_words = set(nltk.corpus.stopwords.words('english')).union(universal_stopwords).union(hr_fluff)

def clean_text(raw_text):
    if not isinstance(raw_text, str) or not raw_text:
        return ""

    text = raw_text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    # HANYA hapus karakter khusus yang benar-benar tidak berguna, pertahankan titik dan plus (C++, Node.js)
    text = re.sub(r'[^\w\s\.\+]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    doc = nlp(text)
    purified_tokens = [
        token.lemma_ for token in doc
        if token.lemma_ not in stop_words and len(token.lemma_) > 1 and not token.text.isnumeric()
    ]

    return " ".join(purified_tokens)

In [18]:
# ==========================================
# EKSEKUSI PIPELINE NLP KE DATASET
# ==========================================
print("1. Menggabungkan kolom teks menjadi satu dokumen utuh...")
clean_df['Combined_Text'] = clean_df['Title'] + " " + \
                            clean_df['JobDescription'].fillna('') + " " + \
                            clean_df['JobRequirment'].fillna('') + " " + \
                            clean_df['RequiredQual'].fillna('')

print("2. Menerapkan NLP Protocol (clean_text) ke seluruh dataset...")
tqdm.pandas(desc="Membersihkan Teks")
clean_df['Cleaned_Text'] = clean_df['Combined_Text'].progress_apply(clean_text)
print("\n PRA-PEMROSESAN SELESAI ")

1. Menggabungkan kolom teks menjadi satu dokumen utuh...
2. Menerapkan NLP Protocol (clean_text) ke seluruh dataset...


Membersihkan Teks:   0%|          | 0/12379 [00:00<?, ?it/s]


 PRA-PEMROSESAN SELESAI 


---
## MINGGU 3: PEMBANGUNAN MODEL (MACHINE LEARNING)

### Tugas 3.1: Ekstraksi Fitur & Matriks Kemiripan
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Mengubah teks bersih menjadi vektor numerik menggunakan algoritma **TF-IDF**. Kemudian membangun fungsi untuk menghitung **Cosine Similarity** antara vektor CV dengan matriks vektor lowongan kerja. Terapkan MLflow tracking jika bereksperimen dengan parameter (max_features, n-grams).

In [19]:
def extract_yoe(text):
    """
    Ekstraksi Years of Experience secara heuristik dari teks.
    """
    text = str(text).lower()
    pattern = r'(\d+)\s*(?:\+|-)?\s*(?:to\s*\d+\s*)?years?(?:\s*of\s*experience)?'
    matches = re.findall(pattern, text)
    if matches:
        return max([int(m) for m in matches])
    return 0

def boost_and_align_keywords(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()

    # Kumpulan kamus jembatan istilah berdasarkan domain bidang
    synonyms = {
        # --- DOMAIN 1: TECH & AI ---
        r'\b(prompt engineering|nlp|deep learning|ai|machine learning|tensor flow|scikit-learn|classification|sentiment analysis)\b': 'statistics data_science analytics',
        r'\b(next\.js|react|frontend|full-stack|backend|php|sql|node\.js|framework)\b': 'web_development javascript html css',

        # --- DOMAIN 2: FINANCE & ACCOUNTING ---
        r'\b(pajak|brevet|taxation|tax)\b': 'tax compliance accounting',
        r'\b(bookkeeping|pembukuan|accounting)\b': 'accounting financial_reporting ledger',

        # --- DOMAIN 3: MARKETING & SALES ---
        r'\b(digital marketing|seo|sem|content creator)\b': 'marketing advertising public_relations',
        r'\b(account executive|telemarketing)\b': 'sales retail business_development'
    }

    for pattern, replacement in synonyms.items():
        text = re.sub(pattern, replacement, text)

    return text

def train_and_log_tfidf(df, text_column):
    """
    Pelatihan TF-IDF Vectorizer dengan pelacakan MLflow.
    """
    print("Memulai Eksperimen TF-IDF (Objective Mode)...")
    mlflow.set_experiment("CareerMatch_Objective_TFIDF")

    with mlflow.start_run():
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,
            max_df=0.85
        )

        print("Menyelaraskan Kamus Dataset (Umbrella Terms)...")
        aligned_jobs = df[text_column].apply(boost_and_align_keywords)

        print("Melakukan Vektorisasi...")
        job_matrix = vectorizer.fit_transform(aligned_jobs)

        mlflow.log_param("ngram_range", "(1, 2)")
        mlflow.log_param("vocab_size", len(vectorizer.vocabulary_))

        model_dir = "models"
        os.makedirs(model_dir, exist_ok=True)
        joblib.dump(vectorizer, os.path.join(model_dir, "tfidf_vectorizer.joblib"))
        joblib.dump(job_matrix, os.path.join(model_dir, "job_vectors.joblib"))

        print("Model TF-IDF berhasil dilatih dan disimpan!")
        return vectorizer, job_matrix

def get_job_recommendations(cv_text, tfidf_model, job_matrix, clean_df, top_k=5):
    """
    Pencarian kemiripan dengan optimasi industri terbaik: Title Boosting & Negative Filter.
    Menghilangkan mismatch bidang tanpa merusak integritas data TF-IDF.
    """
    cleaned_cv = clean_text(cv_text)
    aligned_cv = boost_and_align_keywords(cleaned_cv)

    cv_vector = tfidf_model.transform([aligned_cv])
    similarity_scores = cosine_similarity(cv_vector, job_matrix).flatten()

    # Ambil pool kandidat yang cukup besar (Top 100)
    top_100_indices = similarity_scores.argsort()[-100:][::-1]

    cv_yoe = extract_yoe(cv_text)
    temp_recommendations = []

    # Deteksi domain CV pengguna secara otomatis berdasarkan Umbrella Terms
    is_tech_cv = any(kw in aligned_cv for kw in ['web_development', 'data_science_ai', 'cloud_infrastructure'])
    is_finance_cv = any(kw in aligned_cv for kw in ['accounting', 'tax', 'financial_reporting'])
    is_marketing_cv = any(kw in aligned_cv for kw in ['marketing', 'advertising', 'sales'])

    for idx in top_100_indices:
        raw_score = similarity_scores[idx]
        if raw_score < 0.02:
            continue

        job_info = clean_df.iloc[idx]
        job_title = str(job_info['Title']).lower()
        job_yoe = extract_yoe(job_info['RequiredQual'])

        # FILTER KATA KUNCI NEGATIF (ANTI-MISMATCH) ---
        # Jika CV pengguna adalah IT/Tech, singkirkan lowongan yang berbau administrasi fisik/logistik
        if is_tech_cv:
            non_tech_triggers = ['customs', 'logistic', 'warehouse', 'clerical', 'apparel', 'secretary']
            if any(trigger in job_title for trigger in non_tech_triggers):
                continue # Otomatis mengeliminasi lowongan seperti "Customs Specialist"

        # Jika kelak ada CV Keuangan, singkirkan lowongan IT yang tidak relevan
        if is_finance_cv:
            tech_triggers = ['developer', 'programmer', 'software engineer', 'sysadmin', 'devops']
            if any(trigger in job_title for trigger in tech_triggers):
                continue

        # SENSOR KEDUA: ATURAN LEVEL PENGALAMAN (YOE) ---
        if cv_yoe == 0 and any(kw in job_title for kw in ['senior', 'lead', 'architect', 'principal']):
            continue
        if cv_yoe < job_yoe:
            continue

        # SENSOR KETIGA: TITLE BOOSTING (BONUS SKOR INTUITIF) ---
        title_boost = 1.0

        # Berikan bonus skor 15% jika judul lowongan sesuai dengan rumpun keahlian utama di CV
        if is_tech_cv and any(kw in job_title for kw in ['developer', 'analyst', 'engineer', 'programmer', 'it', 'technical']):
            title_boost = 1.15
        elif is_finance_cv and any(kw in job_title for kw in ['accountant', 'finance', 'audit', 'tax', 'bookkeeper']):
            title_boost = 1.15
        elif is_marketing_cv and any(kw in job_title for kw in ['marketing', 'sales', 'seo', 'content', 'brand']):
            title_boost = 1.15

        scaled_score = np.sqrt(raw_score)
        final_score = (scaled_score * 2.0 * title_boost) * 100
        final_score = min(95.0, final_score)

        temp_recommendations.append({
            'Job Title': job_info['Title'],
            'Match Score': final_score,
            'Required YoE': job_yoe,
            'Required Qualifications': job_info['RequiredQual']
        })

    # Urutkan kembali berdasarkan hasil akhir setelah dilakukan pembobotan judul
    sorted_recommendations = sorted(temp_recommendations, key=lambda x: x['Match Score'], reverse=True)

    final_results = []
    for rec in sorted_recommendations[:top_k]:
        rec['Match Score'] = round(rec['Match Score'], 2)
        final_results.append(rec)

    return final_results

In [24]:
try:
    tfidf_model, job_matrix = train_and_log_tfidf(clean_df, 'Cleaned_Text')

    print("\n" + "="*50)
    print(" SIMULASI INFERENCE CAREERMATCH AI (COLAB ENVIRONMENT)")
    print("="*50)
    print(" Silakan upload file CV Anda (.pdf)...")

    uploaded = colab_files.upload()

    if uploaded:
        filename = next(iter(uploaded))
        print(f"\n File '{filename}' berhasil diunggah!")

        print(" Mengekstrak teks dari PDF...")
        raw_cv_text, status = extract_pdf_text(filename)

        if raw_cv_text:
            print(f" Terdeteksi Pengalaman di CV: ~{extract_yoe(raw_cv_text)} Tahun")
            print(" Menghitung matriks kecocokan dengan database...")

            hasil_rekomendasi = get_job_recommendations(
                cv_text=raw_cv_text,
                tfidf_model=tfidf_model,
                job_matrix=job_matrix,
                clean_df=clean_df,
                top_k=5
            )

            if hasil_rekomendasi:
                for idx, rec in enumerate(hasil_rekomendasi, 1):
                    print(f"\n Rekomendasi {idx}:")
                    print(f" Posisi      : {rec['Job Title']}")
                    print(f" Match Score : {rec['Match Score']}%")
                    print(f" Syarat (YoE): {rec['Required YoE']} Tahun")
                    print(f" Kualifikasi : {rec['Required Qualifications'][:300]}...\n")
            else:
                print(" Sistem tidak menemukan lowongan yang cocok.")
        else:
            print(f" Gagal mengekstrak teks PDF: {status}")
    else:
        print(" Tidak ada file yang diunggah. Simulasi dibatalkan.")

except Exception as e:
    print(f"\n ERROR : {e}")

Memulai Eksperimen TF-IDF (Objective Mode)...
Menyelaraskan Kamus Dataset (Umbrella Terms)...
Melakukan Vektorisasi...
Model TF-IDF berhasil dilatih dan disimpan!

 SIMULASI INFERENCE CAREERMATCH AI (COLAB ENVIRONMENT)
 Silakan upload file CV Anda (.pdf)...


Saving Resume_Armand Al-Farizy.pdf to Resume_Armand Al-Farizy.pdf

 File 'Resume_Armand Al-Farizy.pdf' berhasil diunggah!
 Mengekstrak teks dari PDF...
 Terdeteksi Pengalaman di CV: ~0 Tahun
 Menghitung matriks kecocokan dengan database...

 Rekomendasi 1:
 Posisi      : Data Analyst/ Technical Writer
 Match Score : 78.67%
 Syarat (YoE): 0 Tahun
 Kualifikasi : - Strong understanding of data mining models, structures, theories,
principles, and practices;
- Strong familiarity with data preparation, processing and
classification;
- Working technical experience with relational databases and SQL;
- Good knowledge of data modeling tools such as SAIKU, Pent...


 Rekomendasi 2:
 Posisi      : Perl Developer
 Match Score : 68.98%
 Syarat (YoE): 0 Tahun
 Kualifikasi : - Extensive experience in object-oriented perl programming;
- Basic SQL/RDBMS experience;
- Advanced knowledge of and experience developing under Unix/Linux,
including shell scripting.
Desirable Qualifications:
- Experience using 

---
## MINGGU 4: PENGEMBANGAN ANTARMUKA (DEPLOYMENT)

### Tugas 4.1: Desain & Logika Aplikasi Streamlit
**PIC:** Islahul (UI/UX) dibantu oleh Armand (Integrasi Model)

**Deskripsi Tugas:** Merancang tata letak aplikasi web (Front-end). *Catatan: Kode untuk Streamlit biasanya ditulis di file `app.py` terpisah, namun cell ini dapat digunakan untuk membuat prototipe UI sederhana atau fungsi render.*

In [ ]:
# Area eksperimen prototipe logika Streamlit UI
from IPython.display import display, HTML

# Prototipe render card rekomendasi
def render_job_card(rank, title, score, yoe, qualifications):
    """Fungsi render card — akan diadaptasi ke Streamlit di app.py"""
    color = '#4ade80' if score >= 80 else ('#c5f467' if score >= 50 else '#f87171')
    display(HTML(f"""
    <div style="background:#111811;border:1px solid #1f2e1f;border-radius:12px;
                padding:16px;margin-bottom:10px;font-family:sans-serif;">
        <div style="display:flex;justify-content:space-between;">
            <span style="color:#e8e8e8;font-weight:600;">#{rank} {title}</span>
            <span style="color:{color};font-weight:700;">{score}%</span>
        </div>
        <div style="color:#8a9a8a;font-size:0.8rem;margin-top:6px;">
            YoE: {yoe} thn &nbsp;|&nbsp; {qualifications[:200]}...
        </div>
    </div>
    """))

# --- Test prototipe dengan data dummy ---
render_job_card(1, "Data Analyst", 78.5, 0, "Strong understanding of SQL, Python, data visualization...")
render_job_card(2, "PHP Developer", 65.3, 1, "Experience in PHP, MySQL, HTML/CSS, JavaScript...")

---
## MINGGU 5: PENGUJIAN & QUALITY ASSURANCE

### Tugas 5.1: Pengujian Batas (Edge Cases)
**PIC:** Faber (QA Tester)

**Deskripsi Tugas:** Menguji sistem dengan mengunggah berbagai jenis skenario (CV kosong, CV penuh gambar/tidak terbaca PyPDF2, CV bahasa campuran, CV dengan format aneh).

> **Saran:**
> Dokumentasikan setiap *error* yang muncul di cell ini dan laporkan ke LeadAI Engineer agar fungsi Python-nya bisa diperbaiki (*error handling*).

In [ ]:
# Tulis kode pengujian atau buat log testing menggunakan skenario khusus di sini